In [16]:
import ast
import math
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
import pandas as pd
import pickle
import re

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gowtham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# RecipeNLG Dataset

### Create the sample dataset

In [ ]:
# Read the dataset
df = pd.read_csv("../dataset/full_dataset.csv")
print(df.head()), print(len(df))

In [ ]:
# Get the columns
df.columns

In [ ]:
# Drop the Unnamed: 0 column
df.drop(columns=["Unnamed: 0"], inplace=True)
print(df.head())

In [ ]:
# Get a sample of 5000 recipes
sample_df = df.sample(n=5000, random_state=42)
print(len(sample_df))
sample_df.to_csv("../dataset/sample_dataset.csv", index=False)

### Prepare the dataset

In [ ]:
# Read the sample dataset
import pandas as pd

df = pd.read_csv("../dataset/sample_dataset.csv")
print(df.head())

In [ ]:
# Combine the string
import json

def combine_string(row):
    raw_ingredients_array = json.loads(row['ingredients'])
    raw_directions_array = json.loads(row['directions'])
    raw_ner_array = json.loads(row['NER'])

    return (
        f"title {row['title']}\n"
        f"{', '.join(raw_ingredients_array)}\n"
        f"{', '.join(raw_directions_array)}\n"
        f"{', '.join(raw_ner_array)}"
    ).lower()

df["combined"] = df.apply(combine_string, axis=1)
print(df["combined"].head())

### Text Pre-processing

In [ ]:
# Clean recipe text
def clean_recipe_text(text):
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove fractions and numbers
    text = re.sub(r'\d+/\d+', '', text) # Remove fractions
    text = re.sub(r'\d+\.\d+', '', text) # Remove decimals
    text = re.sub(r'\d+', '', text) # Remove numbers

    # 3. Define a list of units to remove
    noise_list = [
        # Units
        'lb', 'lbs', 'pound', 'pounds', 'oz', 'ounce', 'ounces', 
        'gram', 'grams', 'kg', 'kg', 'g', 'ml', 'clove', 'cloves', 'c',
        'cup', 'cups', 'tsp', 'teaspoon', 'teaspoons', 'tbsp', 
        'tablespoon', 'tablespoons', 'can', 'cans', 'package', 'pkgs',
        # Prep Verbs
        'diced', 'minced', 'chopped', 'sliced', 'shredded', 'grated', 'crushed', 
        'mashed', 'pureed', 'ground', 'slivered', 'halved', 'quartered', 'cubed',
        'grnd',
        # States & Descriptors
        'frozen', 'chilled', 'melted', 'fresh', 'dried', 'cooked', 'raw', 
        'large', 'small', 'medium', 'optional', 'divided', 'drained', 'rinsed'
    ]

    # 4. Create a regex pattern from the noise list
    noise_pattern = r'\b(' + '|'.join(noise_list) + r')\b' # \b is a word boundary to avoid partial matches
    text = re.sub(noise_pattern, '', text)

    # 5. Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # 6. Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['cleaned'] = df['combined'].apply(clean_recipe_text)
print(df['cleaned'].head())

In [ ]:
# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['cleaned'] = df['cleaned'].apply(remove_stopwords)
print(df['cleaned'].head())

In [ ]:
# Using Snowball Stemmer
stemmer = SnowballStemmer('english')

df['cleaned'] = df['cleaned'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

### Create the Vector Space

In [ ]:
# Build the vocabulary
vocabulary = set()
for doc in df['cleaned']:
    for word in doc.split():
        vocabulary.add(word)

print(len(vocabulary))

In [ ]:
# Calculate the term frequency matrix
### TF(t, d) = Number of times term t appears in document d / Total number of terms in document d
def calculate_term_frequency_matrix(text):
    all_terms = text.split()
    total_terms = len(all_terms)
    term_freq = {}
    for term in all_terms:
        if term in term_freq:
            term_freq[term] += 1/total_terms
        else:
            term_freq[term] = 1/total_terms
    return term_freq

df['term_frequency'] = df['cleaned'].apply(calculate_term_frequency_matrix)


In [ ]:
# Calculate the inverse document frequency
### IDF(t, D) = log(Total number of documents / Number of documents containing term t)
inverse_document_frequency = {}
total_documents = len(df)

for term in vocabulary:
    count = 0
    for row in df['cleaned']:
        if term in row:
            count += 1
    inverse_document_frequency[term] = math.log(total_documents / count)

In [ ]:
# Calculate the TD-IDF matrix
### TF-IDF(t, d, D) = TF(t, d) * IDF(t, D)
def calculate_tf_idf_matrix(dictionary):
    tfidf = dict()
    for key, value in dictionary.items():
        tfidf[key] = value * inverse_document_frequency[key]
    return tfidf

df['tfidf'] = df['term_frequency'].apply(calculate_tf_idf_matrix)

### Saving the new data

In [ ]:
# Save the dataframe
df.to_csv("../data/5000_data_with_tf_idf.csv", index=False)

In [ ]:
# Save vocabulary
pickle.dump(vocabulary, open("../data/vocabulary.pkl", "wb"))

In [ ]:
# Save IDF values
pickle.dump(inverse_document_frequency, open("../data/inverse_document_frequency.pkl",  "wb"))

### Process new queries

In [9]:
# Fetch test data, vocabulary and IDF matrix
data = pd.read_csv("../data/5000_data_with_tf_idf.csv")
vocab = pickle.load(open("../data/vocabulary.pkl", "rb"))
idf_matrix = pickle.load(open("../data/inverse_document_frequency.pkl", "rb"))

In [13]:
# GET ALL UTILS
## Clean the data from noise
def clean_data(text: str) -> str:
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove fractions and numbers
    text = re.sub(r'\d+/\d+', '', text) # Remove fractions
    text = re.sub(r'\d+\.\d+', '', text) # Remove decimals
    text = re.sub(r'\d+', '', text) # Remove numbers

    # 3. Define a list of units to remove
    noise_list = [
        # Units
        'lb', 'lbs', 'pound', 'pounds', 'oz', 'ounce', 'ounces', 
        'gram', 'grams', 'kg', 'kg', 'g', 'ml', 'clove', 'cloves', 'c',
        'cup', 'cups', 'tsp', 'teaspoon', 'teaspoons', 'tbsp', 
        'tablespoon', 'tablespoons', 'can', 'cans', 'package', 'pkgs',
        # Prep Verbs
        'diced', 'minced', 'chopped', 'sliced', 'shredded', 'grated', 'crushed', 
        'mashed', 'pureed', 'ground', 'slivered', 'halved', 'quartered', 'cubed',
        'grnd',
        # States & Descriptors
        'frozen', 'chilled', 'melted', 'fresh', 'dried', 'cooked', 'raw', 
        'large', 'small', 'medium', 'optional', 'divided', 'drained', 'rinsed'
    ]

    # 4. Create a regex pattern from the noise list
    noise_pattern = r'\b(' + '|'.join(noise_list) + r')\b' # \b is a word boundary to avoid partial matches
    text = re.sub(noise_pattern, '', text)

    # 5. Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # 6. Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

## Remove stopwords
stop_words = set(stopwords.words('english'))
def remove_stopwords(text: str) -> str:
    return ' '.join([word for word in text.split() if word not in stop_words])

## Calculate TF
def calculate_term_frequency(text: str) -> dict:
    all_terms = text.split()
    total_terms = len(all_terms)
    term_freq = {}
    for term in all_terms:
        if term in term_freq:
            term_freq[term] += 1/total_terms
        else:
            term_freq[term] = 1/total_terms
    return term_freq

## Calculate TFIDF
def calculate_tfidf(dictionary: dict) -> dict:
    tfidf = dict()
    for key, value in dictionary.items():
        idf_value = idf_matrix.get(key, 0)
        tfidf[key] = value * idf_value
    return tfidf

In [14]:
# Create the querying function and pickle it as well
def process_query(query):
    # Clean the query
    noise_cleaned_data = clean_data(query)

    # Remove the stopwords
    stopword_removed_data = remove_stopwords(noise_cleaned_data)

    # Calculate term frequency
    term_frequency = calculate_term_frequency(stopword_removed_data)

    # Calculate tfidf
    tfidf_vector = calculate_tfidf(term_frequency)
    return tfidf_vector

In [15]:
query = "Spicy chicken"

processed_query = process_query(query)
processed_query

{'spicy': 0.0, 'chicken': 1.0049577395156128}

In [18]:
# Cosine Similarity
## Calculate magnitude
def calculate_magnitude(tfidf_dict: dict) -> float:
    if not tfidf_dict:
        return 0
    sum_squares = sum(value ** 2 for value in tfidf_dict.values())
    return math.sqrt(sum_squares)

## Calculate dot product
def calculate_dot_product(query_vec: dict, recipe_vec: dict) -> float:
    dot_product = 0
    for word, weight in query_vec.items():
        if word in recipe_vec:
            dot_product += recipe_vec[word] * weight

    return dot_product

## Calculate cosine similarity
def get_similarity(query_vec: dict, recipe_vec: dict) -> float:
    # 1, Dot product
    dot_product = calculate_dot_product(query_vec, recipe_vec)

    # 2. Magnitude
    mag_query = calculate_magnitude(query_vec)
    mag_recipe = calculate_magnitude(recipe_vec)

    # 3. Prevent division by 0
    if mag_query == 0 or mag_recipe == 0:
        return 0

    return dot_product

In [19]:
def run_search(query, df):
    # 1. Process query into a vector
    query_vec = process_query(query)

    # 2. Convert string_dict back to real dicts if necessary
    if isinstance(df['tfidf'].iloc[0], str):
        df['tfidf'] = df['tfidf'].apply(ast.literal_eval)

    # 3. Calculate similarity for every row
    df['score'] = df['tfidf'].apply(
        lambda recipe_vec: get_similarity(query_vec, recipe_vec)
    )
    
    # 4. Sort and return top 5 results
    return df.sort_values(by='score', ascending=False).head(5)

run_search("Chicken", data)

,title,ingredients,directions,link,source,NER,combined,cleaned,term_frequency,tfidf,score
4203,Chicken And Rice Soup(Greek: Avgolemomo),"[""1 stewing chicken"", ""6 c. broth"", ""1 c. rice...","[""Boil chicken until cooked."", ""Remove chicken...",www.cookbooks.com/Recipe-Details.aspx?id=892477,Gathered,"[""chicken"", ""broth"", ""rice"", ""eggs"", ""lemon"", ...",title chicken and rice soup(greek: avgolemomo...,titl chicken rice soupgreek avgolemomo stew ch...,"{'titl': 0.03225806451612903, 'chicken': 0.193...","{'titl': 0.0, 'chicken': 0.389015899167334, 'r...",0.781889
3209,Chicken Pot Pie,"[""3 large chicken breasts"", ""3 boiled eggs"", ""...","[""Boil chicken until tender and debone."", ""Pla...",www.cookbooks.com/Recipe-Details.aspx?id=309282,Gathered,"[""chicken breasts"", ""eggs"", ""cream of chicken ...",title chicken pot pie\n3 large chicken breasts...,titl chicken pot pie chicken breast boil egg c...,"{'titl': 0.015625, 'chicken': 0.1875, 'pot': 0...","{'titl': 0.0, 'chicken': 0.3768591523183548, '...",0.757455
592,Chicken Casserole,"[""Pepperidge Farm dressing/stuffing mix"", ""6 l...","[""Reserve chicken broth."", ""Boil chicken in wa...",www.cookbooks.com/Recipe-Details.aspx?id=1064785,Gathered,"[""chicken"", ""mushroom soup"", ""cream of chicken...",title chicken casserole\npepperidge farm dress...,titl chicken casserol pepperidg farm dressings...,"{'titl': 0.01694915254237288, 'chicken': 0.186...","{'titl': 0.0, 'chicken': 0.3747300045651438, '...",0.753176
1153,Italian Chicken,"[""4 boneless chicken breasts"", ""1/2 c. butter""...","[""Slice chicken into 1 x 1-inch pieces."", ""Sau...",www.cookbooks.com/Recipe-Details.aspx?id=638893,Gathered,"[""chicken breasts"", ""butter"", ""Italian bread c...",title italian chicken\n4 boneless chicken brea...,titl italian chicken boneless chicken breast b...,"{'titl': 0.016666666666666666, 'italian': 0.05...","{'titl': 0.0, 'italian': 0.1627251501255676, '...",0.740623
1654,Chicken Casserole,"[""1 chicken"", ""1/2 c. milk"", ""1 can cream of c...","[""Stew and debone chicken."", ""Save broth."", ""S...",www.cookbooks.com/Recipe-Details.aspx?id=231753,Gathered,"[""chicken"", ""milk"", ""cream of chicken soup"", ""...","title chicken casserole\n1 chicken, 1/2 c. mil...",titl chicken casserol chicken milk cream chick...,"{'titl': 0.017857142857142856, 'chicken': 0.17...","{'titl': 0.0, 'chicken': 0.3589134783984331, '...",0.721386
